# WER Challenge

## Inhaltsverzeichnis

- [Vorwort](#vorwort)
- [Einleitung](#einleitung)
- [Definition der Fragestellung](#definition-der-fragestellung)
- [Hypothese](#hypothese)
- [Lösungsstrategie](#lösungsstrategie)
- [Einlesen von Dateien und kombinieren](#einlesen-von-dateien-und-kombinieren)
- [Systemdefinition](#systemdefinition)
- [Korrelations Plot](#korrelations-plot)
- [Analyse der Korrelationen](#analyse-der-korrelationen)
  - [Die Gesamtbewoelkung](#die-gesamtbewoelkung)
  - [Die Luftfeuchtigkeit](#die-luftfeuchtigkeit)
  - [Der Luftdruck](#der-luftdruck)
  - [Der Scope](#der-scope)
  - [Die Jahreszeiten - Faktor Zeit](#die-jahreszeiten---faktor-zeit)
- [Wahrscheinlichkeitsmodell](#wahrscheinlichkeitsmodell)
- [Abhängigkeiten und Unsicherheiten](#abhaengigkeiten-und-unsicherheiten)
- [Erweiterung durch neue Erkenntnisse](#erweiterung-durch-neue-erkenntnisse)

# Vorwort

Auf dieser Open Mini Challenge haben sich Michaela Brutenic, Roberto Fazekas und Emircan Bagcivan beteiligt.

# Einleitung
Die vorliegende Open Mini Challenge hat als Ziel die Entwicklung eines Wahrscheinlichkeitsmodells, das vorhersagt, ob es am nächsten Tag in Basel regnen wird. Grundlage für die Modellierung bilden regionale Wetterdaten aus den Jahren 2000 bis 2025, die eine Vielzahl meteorologischer Merkmale wie Temperatur, Luftfeuchtigkeit, Luftdruck, Windverhältnisse und Niederschlagsmengen enthalten.

Ziel ist es, auf Basis dieser Daten mithilfe klassischer Methoden der Wahrscheinlichkeitsrechnung – wie etwa bedingter Wahrscheinlichkeiten, Häufigkeitsverteilungen, Übergangswahrscheinlichkeiten oder auch einfacher stochastischer Modelle – ein mathematisches Modell zu konstruieren, dass die Regenwahrscheinlichkeit für den folge Tag beschreibt.

Die vollständige Datenlage erlaubt dabei eine fundierte Analyse, um die Abhängigkeiten zwischen Wettermerkmalen und Regenereignissen zu untersuchen. Das Modell soll dabei nicht nur eine Ja/Nein-Aussage treffen,  [sondern explizit die Wahrscheinlichkeit eines Regenereignisses angeben.]  Es bietet somit einen praktischen Anwendungsfall für stochastische Denkweisen im realweltlichen Kontext der Wettervorhersage.

Die Daten liegen in strukturierter Form vor und ermöglichen sowohl eine fundierte explorative Analyse als auch die Anwendung klassischer Verfahren der Wahrscheinlichkeitsrechnung. Ziel ist es, ein robustes, nachvollziehbares Modell zu entwickeln, das verlässliche Aussagen zur Regenwahrscheinlichkeit liefert 

# Definition der Fragestellung
Im Zentrum dieser Arbeit steht die folgende Fragestellung:

"Wie hoch ist die Wahrscheinlichkeit für ein Regenereignis am folgenden Tag in Basel basierend auf historischen Wetterdaten?"

Dabei sollen insbesondere stochastische Zusammenhänge zwischen verschiedenen meteorologischen Einflussgrössen und dem Auftreten von Regenereignissen identifiziert und in ein Wahrscheinlichkeitsmodell überführt werden. Das Ziel besteht darin, ein Modell zu entwickeln, das auf klar definierten, datenbasierten Regeln beruht und eine quantifizierte Vorhersage zur Regenwahrscheinlichkeit erlaubt – anstelle einer binären Entscheidung. Was unsere Fragestellung klar zu einer probabilistischen Fragestellung macht. Wir interessieren uns nicht für eine klare Ja / Nein Aussage, sondern für die Wahrscheinlichkeit - zu wie viel
Prozent es regnet.

# Hypothese
Es besteht ein signifikanter Zusammenhang zwischen bestimmten meteorologischen Variablen (z.B. Luftfeuchtigkeit, Bewölkung) und der Wahrscheinlichkeit des
Auftretens von Niederschlag. Ein geeignetes Wahrscheinlichkeitsmodell kann verwendet werden, um diesen Zusammenhang zu quantifizieren und zu prognostizieren.


# Lösungsstrategie
überblick über die Daten
Korrealtionen angeschaut
recherche
Hypothesentest durchgeführt
Verteilungen für die ausgewählte Zufallvar 
Monte-carlo-simula
dokumentiert

# Einlesen der Datei

**Imports**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, mannwhitneyu
import distfit as distfit
from IPython.display import display
import seaborn as sns
from scipy import stats
import numpy as np
from datetime import datetime, timedelta


**Einlesen**

In [ ]:
df_2000_2025 = pd.read_csv('data/wetterdaten_2000_2025.csv', sep=';')

# Umwandeln des Datums
df_2000_2025['Datum'] = pd.to_datetime(df_2000_2025['Datum'])

### Allgemeine Übersicht über die Daten
Ein grober Überblick über den Datensatz ist essentiell, damit wir wissen, um welche Art von Beobachtungspunkten es sich handelt.

In [ ]:
df_2000_2025.info()
df_2000_2025.describe()
print("-------------")
print(df_2000_2025.isna().sum())

#### Bearbeitung der Daten für die Analyse
Um die Regenwahrscheinlichkeit für den nächsten Tag zu berechnen, müssen wir die Niederschlagsmenge von morgen als Zielvariable verwenden. Dafür verschieben wir die Niederschlagsdaten um einen Tag nach vorne, damit sie zur jeweils aktuellen Zeile passen. Diese zeitliche Verschiebung ist zwar für Vorhersagemodelle sinnvoll, kann aber bei anderen Analysen, z.B. bei der Untersuchung von Zusammenhängen in der Originalzeitreihe, zu Problemen führen.

Deshalb speichern wir die verschobenen Daten in einer separaten Excel-Datei. So können wir je nach Analyse entweder mit den Originaldaten oder den verschobenen Daten arbeiten. Das hilft uns, Fehler in der Analyse zu vermeiden und flexibel zwischen verschiedenen Ansätzen zu wechseln.

Dieses Vorgehen orientiert sich an gängigen Praktiken im Feature Engineering bei Zeitreihendaten (siehe z.Hyndman, R. J., & Athanasopoulos, G. (2018). Forecasting: Principles and Practice (2nd ed.). OTexts. https://otexts.com/fpp2/). Quelle Daten!

In [ ]:
df_mit_shift = df_2000_2025.copy(deep=True)
df_mit_shift['Niederschlag_Morgen'] = df_mit_shift['Niederschlag'].shift(-1)
df_mit_shift = df_mit_shift.drop(columns=['Niederschlag'])
df_mit_shift = df_mit_shift.dropna(subset=['Niederschlag_Morgen'])
df_mit_shift.to_csv('data/mit_shift.csv', index=False, sep=';')

# Systemdefinition

Um unsere Hypothese gezielt untersuchen zu können, ist es sinnvoll, den Umfang der Daten einzugrenzen. Da nicht alle Informationen im Datensatz für unsere Fragestellung relevant sind, benötigen wir eine fundierte Entscheidungsgrundlage dafür, auf welche Variablen wir uns konzentrieren.

Ein geeigneter erster Schritt ist die Analyse der Korrelationen innerhalb des Datensatzes. Dabei interessiert uns insbesondere, welche Variablen am stärksten mit dem Niederschlag zusammenhängen. Zwar erlaubt die Korrelation noch keine Aussage über kausale Zusammenhänge, liefert jedoch wertvolle Hinweise darauf, wo sich ein genauerer Blick lohnt – sie dient also gewissermassen als erste Fährte in der Datenexploration.

#### Definition eines Regentages gemäss WMO-Standard

Für die statistische Analyse von Wahrscheinlichkeitsverteilungen des Niederschlags empfiehlt sich die Verwendung der Definition eines Regentages nach den Standards der Weltorganisation für Meteorologie (WMO). Demnach gilt ein Tag als Regentag, wenn eine tägliche Niederschlagsmenge von mindestens 1,0 mm gemessen wird.

Diese Definition ist international anerkannt und ermöglicht vergleichbare klimatologische Auswertungen über verschiedene geografische Regionen hinweg. Sie stellt eine sinnvolle Abgrenzung dar, da leichtere Niederschläge unterhalb dieser Schwelle oft nicht signifikant zur Gesamtklimabilanz beitragen. (Quelle: https://www.agroorbi.pt/livroagrometeorologia/DocsProg/Temas%26Exerc%C3%ADciosExtraPorCap%C3%ADtulo/Cap1_Introdu%C3%A7%C3%A3o/Docs/WMO%20Guidelines%20on%20the%20Calculation%20of%20Climate%20Normals_en.pdf)


Empfehlung für die Definition eines Regentages gemäß WMO-Standard

Für die statistische Analyse von Wahrscheinlichkeitsverteilungen des Niederschlags wird ein Regentag gemäß den Standards der WMO als ein Tag definiert, an dem eine Niederschlagsmenge von mindestens 1 mm gemessen wird.

Diese Definition ist international anerkannt und ermöglicht vergleichbare klimatologische Auswertungen über verschiedene geografische Regionen hinweg. Sie stellt eine sinnvolle Abgrenzung dar, da leichtere Niederschläge unterhalb dieser Schwelle oft nicht signifikant zur Gesamtklimabilanz beitragen.

In [ ]:
# Kategorisierung der Niederschlagsmenge
def categorize_Regentag(rain_in_mm):
    if rain_in_mm == 0:
        return "Kein Regentag"
    elif 0 < rain_in_mm < 5:
        return "Leichter Regentag"
    else:
        return "Starker Regentag"


df_2000_2025["Regentag_Kategorie"] = df_2000_2025["Niederschlag"].apply(categorize_Regentag)
df_mit_shift["Regentag_Kategorie"] = df_mit_shift["Niederschlag_Morgen"].apply(categorize_Regentag)
df_2000_2025["Regentag_Kategorie"].describe()

In [ ]:
# Häufigkeiten zählen
regen_counts = df_2000_2025["Regentag_Kategorie"].value_counts()

# Balkendiagramm plotten
regen_counts.plot(kind="bar", color="skyblue", edgecolor="black")
plt.title("Häufigkeit der Regenkategorien")
plt.xlabel("Regenkategorie")
plt.ylabel("Anzahl der Tage")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Korrelations Plot

Um die Auswahl der Variablen fundiert treffen zu können, untersuchen wir die Korrelationen zwischen den einzelnen
Beobachtungen. Unsere Wahl fällt dann auf die Beobachtung, die einerseits die höchste Korrelation mit dem
Niederschlag hat und andererseits aus meteorologischer Sicht sinnvoll ist.

In [ ]:
# Zusammenfügen der Daten über die letzten fünf Jahre
df_2000_2025_for_corr = df_mit_shift[[
    'Globalstrahlung in W/m2',
    'Gesamtschneemenge',
    'Gesamtbewölkung',
    'Luftdruck in hPa',
    'Niederschlag_Morgen',
    'Sonnenscheindauer',
    'Tagesmittel Lufttemperatur',
    'Tagesminimum Lufttemperatur',
    'Tagesmaximum Lufttemperatur',
    'Relative Luftfeuchtigkeit'
]].copy(deep=True)

# Korrelationen erkennen
df_correlation = df_2000_2025_for_corr.corr()

plt.figure(figsize=(12, 8))
sns.heatmap(df_correlation, annot=True, cmap='coolwarm', linewidths=.5)
plt.title("Korrelationen des Datasets")
plt.show()

## Analyse der Korrelationen
Zur Verbesserung der Übersichtlichkeit werden die berechneten Korrelationskoeffizienten nach ihrer Stärke in absteigender Reihenfolge sortiert und anschliessend in einem Balkendiagramm (Barplot) visualisiert.

In [ ]:
# Höchste wie tiefste Korrelationen
df_correlation_highest = (df_correlation["Niederschlag_Morgen"].drop("Niederschlag_Morgen")
                          .sort_values(ascending=False)).head(5)
print(df_correlation_highest)
df_correlation_lowes = (df_correlation["Niederschlag_Morgen"]
                        .drop("Niederschlag_Morgen").sort_values(ascending=True)).head(5)

# Zusammenfügen der DF's
combined_corr = (pd.concat([df_correlation_highest, df_correlation_lowes])
                 .sort_values(ascending=False))

# Plotten der Korrelationen
plt.figure(figsize=(10, 6))
combined_corr.plot(kind='bar', color=['green' if corr > 0 else 'red' for corr in combined_corr.values])
plt.title("Top 5 positive und negative Korrelationen mit Niederschlag Morgen")
plt.ylabel("Korrelationskoeffizient")
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### Statistische Tests:
Zur statistischen Überprüfung eines linearen Zusammenhangs zwischen den gewählten Variablen kommt der Spearman-Test zum Einsatz. Der dabei berechnete Korrelationskoeffizient r quantifiziert, sowohl die Richtung als auch die Stärke des linearen Zusammenhangs zwischen zwei metrischen Variablen.

In [ ]:
# Mit Spearman die Variablen herausfinden
results = []
numerical_cols = df_mit_shift.select_dtypes(include='number').columns

for col in numerical_cols:
    if col != 'Niederschlag_Morgen':
        rho, p = spearmanr(df_mit_shift[col], df_mit_shift['Niederschlag_Morgen'])
        results.append({
            'Variable': col,
            'Spearman-Korrelation': rho,
            'p-Wert': p,
            'Signifikant (p<0.05)': p < 0.05
        })

df_results = pd.DataFrame(results)
df_results.sort_values(by='Spearman-Korrelation', ascending=False, inplace=True)
display(df_results)

Beim Blick auf den Korrelations-Plot fällt auf, dass die Gesamtbewölkung sowie der Luftdruck die stärksten Korrelationen mit dem Niederschlag aufweisen – positiv bzw. negativ. Ebenfalls auffällig sind die relative Luftfeuchtigkeit und die Sonnenscheindauer, die im Vergleich zu den übrigen Variablen sowohl relativ hohe als auch niedrige Korrelationswerte zeigen.

Die restlichen Variablen liegen mit ihren Korrelationswerten nahe bei null. Dies bedeutet nicht zwangsläufig, dass kein Zusammenhang besteht – vielmehr lässt sich kein linearer Zusammenhang nachweisen. Nicht-lineare Beziehungen können durch die Pearson-Korrelation nicht abgebildet werden und müssten gegebenenfalls mit anderen Methoden untersucht werden.

### Hypothesen Tests
Um genauer zu überprüfen, ob die ausgewählten Variablen tatsächlich einen Einfluss auf die Regenwahrscheinlichkeit haben, können Hypothesentests herangezogen werden. Die Auswahl eines geeigneten Tests setzt jedoch voraus, dass die Verteilung der beteiligten Variablen im Vorfeld genauer untersucht wird. Insbesondere ist zu prüfen, ob die Variablen normalverteilt sind oder ob alternative Verteilungen vorliegen – da dies direkten Einfluss auf die Wahl zwischen parametrischen und nichtparametrischen Testverfahren hat.

#### Analyse der Verteilung

In [ ]:
for col in ['Luftdruck in hPa', 'Gesamtbewölkung', 'Relative Luftfeuchtigkeit']:
    plt.figure(figsize=(12, 8))

    plt.subplot(1, 2, 1)
    sns.histplot(df_mit_shift[col], kde=True)
    plt.title(f'Histogram {col}')

    plt.subplot(1, 2, 2)
    sns.boxplot(x="Regentag_Kategorie", y=col, data=df_mit_shift)
    plt.title(f'Boxplot: {col}, nach Regenkategorien')

    plt.tight_layout()
    plt.show()

Bevor ein T-Test angewendet werden kann, ist zu überprüfen, ob die zugrunde liegenden Variablen normalverteilt sind. Liegt keine Normalverteilung vor, ist der T-Test nicht geeignet – in diesem Fall kommen alternative, nichtparametrische Testverfahren zum Einsatz, auf die im weiteren Verlauf der Analyse eingegangen wird.


In [ ]:
# Aufteilen in zwei Gruppen Regen oder kein Regen für den ersten Test.
# Neues DataFrame mit Luftfeuchtigkeit + dichotomer Regen-Kategorie
df_regenvergleich = pd.DataFrame(
    {
        'luftfeuchtigkeit': df_mit_shift['Relative Luftfeuchtigkeit'],
        'gesamtbewölkung': df_mit_shift['Gesamtbewölkung'],
        'luftdruck': df_mit_shift['Luftdruck in hPa'],
        'regen_dichotom': df_mit_shift['Regentag_Kategorie'].apply(
            lambda x: 'Regen' if x != 'Kein Regentag' else 'Kein Regen'
        )
    }
)

#### Normalverteilung evaluieren - Luftdruck:
Zur Evaluierung, ob die Verteilung des Luftdrucks der Normalverteilung entspricht, wird ein gängiger Test herangezogen: der Anderson-Darling-Test und der Shapiro-Wilk-Test. Während der Shapiro-Wilk-Test besonders für kleinere Stichproben geeignet ist, bietet sich der Anderson-Darling-Test bei grösseren Datenmengen an, da er empfindlicher auf Abweichungen in den Randbereichen der Verteilung reagiert.

Von allen betrachteten Variablen weist der Luftdruck die Form, die am ehesten einer Glockenkurve entspricht, weshalb er für diese Prüfung ausgewählt wurde.

Die Hypothesen für den Normalverteilungstest lauten:

    Nullhypothese (H₀): Die Variable ist normalverteilt.

    Alternativhypothese (H₁): Die Variable ist nicht normalverteilt.

Analog wird diese Hypothese für folgende Verteilungen angewadt, dies wird bei der Monte-Carlo Simulation wichtig.

Die Entscheidung über die Verwendung eines parametrischen oder nichtparametrischen Hypothesentests im weiteren Verlauf hängt vom Ergebnis dieser Überprüfung ab.
Diese Hypothesen werden für die folgenden Verteilungen vorgenommen:

In [ ]:
verteilungen = {
    'norm': 'Normalverteilung (Gauß-Verteilung)',
    'expon': 'Exponentialverteilung',
    'weibull_min': 'Weibull-Verteilung (Minimum-Variante)',
    'gamma': 'Gammaverteilung',
    'lognorm': 'Log-Normalverteilung',
    'beta': 'Betaverteilung'
}

spalten = ['Luftdruck in hPa', 'Relative Luftfeuchtigkeit', 'Gesamtbewölkung']


def verteilung_bestimmen(spalte):
    dist = distfit.distfit(distr=list(verteilungen.keys()))

    daten = df_mit_shift[spalte].dropna()
    print(f"\n🔍 Beste Verteilung für: {spalte}")
    try:
        dist.fit_transform(daten)

        name = dist.model['name']
        params = dist.model['params']
        score = dist.model['score']

        param_names = ['a', 'b', 'loc', 'scale']

        param_str = ', '.join([f"{n}={p:.4f}" for n, p in zip(param_names, params)])

        print(f"Verteilung: {verteilungen.get(name, name)} ({name})")
        print(f"Parameter: {param_str}")
        print(f"SSE: {score:.6f}")
    except Exception as e:
        print(f"Fehler bei {spalte}: {e}")

    return params


for spalte in spalten:
    verteilung_bestimmen(spalte)

#### Mann-Whitney-U-Test für nicht normalverteilte Variablen
Da die Variable Luftdruck nicht normalverteilt ist, kann der T-Test nicht angewendet werden. Stattdessen eignet sich der Mann-Whitney-U-Test, um zwei unabhängige Gruppen – hier Regentage und Nicht-Regentage – hinsichtlich der kontinuierlichen Zielgrösse Luftdruck zu vergleichen.

Die Hypothesen lauten dabei:

    Nullhypothese (H₀): Die Verteilung des Luftdrucks ist in beiden Gruppen gleich, das heisst, es gibt keinen Unterschied zwischen Regentagen und Nicht-Regentagen.

    Alternativhypothese (H₁): Die Verteilung des Luftdrucks unterscheidet sich signifikant zwischen den Gruppen, was auf einen Einfluss des Luftdrucks auf die Regenwahrscheinlichkeit hinweist.

In [ ]:
gruppe_regen = df_regenvergleich[df_regenvergleich['regen_dichotom'] == 'Regen']['luftdruck']
gruppe_kein = df_regenvergleich[df_regenvergleich['regen_dichotom'] == 'Kein Regen']['luftdruck']

stat, p = mannwhitneyu(gruppe_regen, gruppe_kein, alternative='two-sided')

print(f"Mann-Whitney-U-Test: p = {p:.4f}, U = {stat:.2f}")

if p < 0.05:
    print("→ Signifikanter Unterschied im Luftdruck zwischen Regen und Kein Regen")
else:
    print("→ Kein signifikanter Unterschied im Luftdruck zwischen Regen und Kein Regen")

Ein signifikanter p-Wert spricht dafür, dass sich der Luftdruck an Regentagen und an Tagen ohne Regen signifikant unterscheidet, obwohl die Verteilungen nicht normalverteilt sind. Dieses Ergebnis unterstützt die Annahme, dass der Luftdruck ein potenzieller Einflussfaktor auf das Wettergeschehen ist.

#### Mann-Whitney-U-Test für die anderen Variablen
Da auch die übrigen betrachteten Variablen laut unseren vorherigen Analysen nicht normalverteilt sind, wird für diese ebenfalls der Mann-Whitney-U-Test angewandt. Die jeweiligen Hypothesen lassen sich dabei analog zum Luftdruck formulieren und werden entsprechend an die jeweilige Variable angepasst.

In [ ]:
# Auswahl der Variablen
colums_to_test = ('gesamtbewölkung', 'luftfeuchtigkeit')

# Für die jeweiligen Variablen die Hypothese testen
for col in colums_to_test:
    gruppe_regen = df_regenvergleich[df_regenvergleich['regen_dichotom'] == 'Regen'][col]
    gruppe_kein = df_regenvergleich[df_regenvergleich['regen_dichotom'] == 'Kein Regen'][col]
    stat, p = mannwhitneyu(gruppe_regen, gruppe_kein, alternative='two-sided')

    print(f"Mann-Whitney-U-Test für {col}: p = {p:.4f}, U = {stat:.2f}")

    if p < 0.05:
        print(f"→ Signifikanter Unterschied im {col} zwischen Regen und Kein Regen")
    else:
        print(f"→ Kein signifikanter Unterschied im {col} zwischen Regen und Kein Regen")

### Zusammenhängende Variablen:

Die Analyse des Korrelationsplots zeigt, dass zwischen mehreren Variablen ein deutlicher Zusammenhang besteht. Insbesondere die Temperaturvariablen weisen untereinander eine hohe positive Korrelation auf, was auf einen engen Zusammenhang zwischen diesen Grössen hinweist.

Bei der Gesamtbewölkung ist eine starke negative Korrelation zur Sonnenscheindauer sowie eine mittlere negative Korrelation zur Globalstrahlung zu erkennen. Dieses Ergebnis ist auch meteorologisch plausibel: An Tagen mit langer Sonnenscheindauer ist der Himmel meist weitgehend wolkenfrei, was mit geringer Bewölkung einhergeht.

Darüber hinaus zeigt sich eine mittelstarke negative Korrelation zwischen der Sonnenscheindauer und der Luftfeuchtigkeit sowie zwischen der Sonnenscheindauer und der Globalstrahlung. Auch diese Zusammenhänge lassen sich durch physikalisch-meteorologische Überlegungen nachvollziehen und stützen die Annahme, dass bestimmte Wettergrössen systematisch miteinander verknüpft sind.

### Die Gesamtbewölkung:
##### Zusammenhang zwischen Gesamtbewölkung und Niederschlag:
Eine positive Korrelation zwischen der Häufigkeit von Bewölkung und der Menge des Niederschlags wurde in mehreren Studien festgestellt. So zeigt eine Untersuchung, dass eine Zunahme der Bewölkungshäufigkeit mit einer Zunahme des Niederschlags einhergeht, während bei fehlender Bewölkung kein Niederschlag auftritt Nature (Quelle: Nogueira, M., Barros, A. P., & Bica, R. (2021). A multidecadal climate dataset of precipitation and cloud properties. Scientific Data, 8, 58; https://www.nature.com/articles/s41597-021-01084-6).
Darüber hinaus hängt die Art des Niederschlags oft mit der Wolkenstruktur zusammen: Starke Niederschläge sind häufig mit konvektiven Wolken verbunden, während leichte Niederschläge typischerweise bei niedrigen, stratifizierten Wolkenformen auftreten (Houze, 2014). Dies unterstreicht die Bedeutung der Wolkenhöhe und -art bei der Analyse von Regenereignissen.
##### Einfluss der Gesamtbewölkung auf die Temperatur und den Wasserkreislauf:
Die Bewölkung beeinflusst nicht nur den Niederschlag, sondern auch die Temperaturen in Bodennähe. Tagsüber können Wolken die solare Einstrahlung abschatten, was zu niedrigeren Temperaturen führt. In wolkenlosen Nächten hingegen sinkt die Temperatur nahe dem Boden deutlich, da die langwellige Gegenstrahlung von Wolken fehlt (Quelle: Stephens, G. L. (2005). Cloud Feedbacks in the Climate System: A Critical Review. Journal of Climate, 18(2), 237–273. https://doi.org/10.1175/JCLI-3243.1)
##### Fazit
Die Gesamtbewölkung stellt einen bedeutenden Indikator für das Auftreten von Niederschlag dar, da sie die physikalische Voraussetzung für Regenereignisse bildet. Der Zusammenhang ist jedoch komplex und wird von verschiedenen Faktoren beeinflusst, etwa der Art, Höhe und Dichte der Wolken sowie weiteren meteorologischen Bedingungen wie Luftfeuchtigkeit und Temperatur. Eine hohe relative Luftfeuchtigkeit begünstigt zudem die Wolkenbildung, was die Wahrscheinlichkeit für Niederschlag zusätzlich erhöht.


### Die Luftfeuchtigkeit:
Wenn die relative Luftfeuchtigkeit hoch ist, steigt die Wahrscheinlichkeit, dass der Sättigungspunkt
überschritten wird, sprich es Regnet. Wie viel Wasser die Luft aufnehmen kann hängt dabei von der
Temperatur ab.
(Stadler Form Luftfeuchtigkeit messen, 2023)„Luftfeuchtigkeitswerte beeinflussen Wetterberechnungen z.B. durch die Wasserdampfbilanzgleichung – diese Formel drückt den Einfluss der Luftfeuchtigkeit durch Regen und Kondensation aus und umgekehrt.“(Process Sensing Technologies, 2023)

### Der Luftdruck:
Sinkt der atmosphärische Luftdruck, ist dies häufig ein Hinweis auf das Herannahen eines Tiefdruckgebiets – solche Gebiete sind typischerweise mit feuchter Luft und instabilen Wetterlagen verbunden. In einem Tief steigt die Luft auf, kühlt sich dabei ab, und der in der Luft enthaltene Wasserdampf kondensiert zu Wolken und schließlich zu Niederschlag. Umgekehrt steht hoher Luftdruck meist für stabiles, trockenes Wetter, da die Luft absinkt und sich erwärmt, wodurch Wolkenbildung und Regen unterdrückt werden.

„Tiefdruckgebiete führen in der Regel zu aufsteigender Luft, was zu Abkühlung, Kondensation und folglich zu Niederschlag führt. Hoher Luftdruck hingegen begünstigt absinkende, trockene Luftmassen.“
(Deutscher Wetterdienst, 2022)

„Luftdruckveränderungen sind zentrale Steuerungsmechanismen für Wettersysteme und beeinflussen die Niederschlagswahrscheinlichkeit maßgeblich.“
(MeteoSchweiz, 2023)

### Der Scope:


### Die Jahreszeiten - Faktor Zeit
Da es über die verschiedenen Jahreszeiten bzw. Monaten nicht gleich viel Regnet nehmen wir an, dass dies ebenfalls in unserem Modell berücksichtigt werden sollte, sollte sich ein Muster erkennen lassen. Falls dies nicht so wäre, so ist diese zeitliche Komponente nicht hilfreich für das Bestimmen der Regenwahrscheinlichkeit.

In [ ]:
df_time_niederschlag = df_2000_2025[['Datum', 'Niederschlag']]

df_time_niederschlag.set_index('Datum', inplace=True)

# Jahresliste (optional automatisch aus den Daten extrahieren)
jahre = df_time_niederschlag.index.year.unique()

# Plot pro Jahr
for jahr in sorted(jahre):
    # Daten für das Jahr filtern und monatlich mitteln
    monthly_avg = df_time_niederschlag[df_time_niederschlag.index.year == jahr]['Niederschlag'].resample('ME').mean()

    plt.figure(figsize=(10, 4))
    monthly_avg.plot(marker='o')
    plt.title(f"Monatliche durchschnittliche Niederschlagsmengen – {jahr}")
    plt.xlabel("Monat")
    plt.ylabel("Niederschlag (mm)")
    plt.grid(True)
    plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# Wahrscheinlichkeitsmodell

Hier wird die Monte-Carlo Simulation implementiert mit den Variablen Luftdruck, Luftfeuchtigkeit und Gesamtbewölung

In [ ]:
a_luftdruck, b_luftdruck, loc_luftdruck, scale_luftdruck = verteilung_bestimmen('Luftdruck in hPa')
print(a_luftdruck, b_luftdruck, loc_luftdruck, scale_luftdruck)

a_luftfeuchtigkeit, b_luftfeuchtigkeit, loc_luftfeuchtigkeit, scale_luftfeuchtigkeit = verteilung_bestimmen('Relative Luftfeuchtigkeit')
print(a_luftfeuchtigkeit, b_luftfeuchtigkeit, loc_luftfeuchtigkeit, scale_luftfeuchtigkeit)




In [ ]:
def monte_carlo_regenvorhersage(
    # Parameter für Luftdruck 
    a_luftdruck, b_luftdruck, loc_luftdruck, scale_luftdruck,
    # Parameter für Luftfeuchtigkeit 
    a_luftfeuchtigkeit, b_luftfeuchtigkeit, loc_luftfeuchtigkeit, scale_luftfeuchtigkeit,
    # Anzahl Simulationen
    n_simulationen=10000,
    # Aktuelle Wetterdaten (optional)
    aktueller_luftdruck=None,
    aktuelle_luftfeuchtigkeit=None,
    # Datum für Vorhersage (optional)
    vorhersage_datum=None
):    
    # Datum bestimmen
    heute = datetime.now()
    if vorhersage_datum is None:
        vorhersage_tag = heute + timedelta(days=1)
        zeitraum_text = "morgen"
    else:
        if isinstance(vorhersage_datum, str):
            try:
                vorhersage_tag = datetime.strptime(vorhersage_datum, "%Y-%m-%d")
            except ValueError:
                try:
                    vorhersage_tag = datetime.strptime(vorhersage_datum, "%d.%m.%Y")
                except ValueError:
                    print("⚠️ Ungültiges Datumsformat. Verwende morgen als Standard.")
                    vorhersage_tag = heute + timedelta(days=1)
        else:
            vorhersage_tag = vorhersage_datum
        
        # Zeitraum-Text bestimmen
        tage_differenz = (vorhersage_tag - heute).days
        if tage_differenz == 0:
            zeitraum_text = "heute"
        elif tage_differenz == 1:
            zeitraum_text = "morgen"
        
    
    # Wochentag auf Deutsch
    wochentage = ['Montag', 'Dienstag', 'Mittwoch', 'Donnerstag', 'Freitag', 'Samstag', 'Sonntag']
    wochentag = wochentage[vorhersage_tag.weekday()]
    
    print(f"🎲 Starte Monte-Carlo-Simulation mit {n_simulationen:,} Iterationen...")
    print("=" * 70)
    print(f"📅 Heute: {heute.strftime('%A, %d.%m.%Y')} um {heute.strftime('%H:%M')} Uhr")
    print(f"🎯 Vorhersage für: {wochentag}, {vorhersage_tag.strftime('%d.%m.%Y')} ({zeitraum_text})")
    print("=" * 70)
    
    # Arrays für Ergebnisse
    regen_wahrscheinlichkeiten = []
    simulierte_luftdruck = []
    simulierte_luftfeuchtigkeit = []
    
    # Monte-Carlo-Simulationen
    for i in range(n_simulationen):
        # Zufällige Werte aus den bestimmten Verteilungen generieren
        if aktueller_luftdruck is None:
            # Luftdruck aus Verteilung sampeln (z.B. beta-Verteilung)
            luftdruck = stats.beta.rvs(a_luftdruck, b_luftdruck, 
                                     loc=loc_luftdruck, scale=scale_luftdruck)
        else:
            # Leichte Variation um aktuellen Wert
            luftdruck = np.random.normal(aktueller_luftdruck, 2.0)
        
        if aktuelle_luftfeuchtigkeit is None:
            # Luftfeuchtigkeit aus Verteilung sampeln
            luftfeuchtigkeit = stats.beta.rvs(a_luftfeuchtigkeit, b_luftfeuchtigkeit,
                                            loc=loc_luftfeuchtigkeit, scale=scale_luftfeuchtigkeit)
        else:
            # Leichte Variation um aktuellen Wert
            luftfeuchtigkeit = np.random.normal(aktuelle_luftfeuchtigkeit, 5.0)
        
        # Regenwahrscheinlichkeit basierend auf Wetterbedingungen berechnen
        # Diese Logik basiert auf meteorologischen Zusammenhängen
        regen_score = berechne_regen_score(luftdruck, luftfeuchtigkeit)
        
        # Speichern für Analyse
        regen_wahrscheinlichkeiten.append(regen_score)
        simulierte_luftdruck.append(luftdruck)
        simulierte_luftfeuchtigkeit.append(luftfeuchtigkeit)
    
    # Ergebnisse analysieren
    regen_wahrscheinlichkeiten = np.array(regen_wahrscheinlichkeiten)
    
    # Wahrscheinlichkeit für Regen (Score > 0.5)
    regen_ja = np.sum(regen_wahrscheinlichkeiten > 0.5)
    regen_prozent = (regen_ja / n_simulationen) * 100
    
    # Statistiken
    mittlere_wahrscheinlichkeit = np.mean(regen_wahrscheinlichkeiten) * 100
    std_wahrscheinlichkeit = np.std(regen_wahrscheinlichkeiten) * 100
    
    # Ergebnisse ausgeben
    print(f"\n🌧️  REGENVORHERSAGE FÜR BASEL - {wochentag.upper()}, {vorhersage_tag.strftime('%d.%m.%Y')}")
    print("=" * 70)
    print(f"📊 Anzahl Simulationen: {n_simulationen:,}")
    print(f"☔ Wahrscheinlichkeit für Regen {zeitraum_text}: {regen_prozent:.1f}%")
    print(f"📈 Mittlere Regenwahrscheinlichkeit: {mittlere_wahrscheinlichkeit:.1f}% ± {std_wahrscheinlichkeit:.1f}%")
    
    # Konfidenzintervall
    percentile_25 = np.percentile(regen_wahrscheinlichkeiten, 25) * 100
    percentile_75 = np.percentile(regen_wahrscheinlichkeiten, 75) * 100
    print(f"50% der Simulationen: {percentile_25:.1f}% - {percentile_75:.1f}%")
    
    # Interpretation mit Datum
    if regen_prozent > 70:
        print(f"Prognose für {zeitraum_text}: HOHE Regenwahrscheinlichkeit - Regenschirm empfohlen!")
    elif regen_prozent > 40:
        print(f"Prognose für {zeitraum_text}: MODERATE Regenwahrscheinlichkeit - Regenschirm zur Sicherheit")
    else:
        print(f"Prognose für {zeitraum_text}: NIEDRIGE Regenwahrscheinlichkeit - Schönes Wetter erwartet")
    
    # Zusätzliche Zeitinformationen
    if (vorhersage_tag - heute).days > 3:
        print(f"Hinweis: Langzeitprognose für {(vorhersage_tag - heute).days} Tage - Unsicherheit steigt!")
    
    # Visualisierung
    erstelle_visualisierung(regen_wahrscheinlichkeiten, simulierte_luftdruck, 
                          simulierte_luftfeuchtigkeit, regen_prozent, 
                          vorhersage_tag, zeitraum_text)
    
    return {
        'vorhersage_datum': vorhersage_tag,
        'wochentag': wochentag,
        'zeitraum_text': zeitraum_text,
        'regen_wahrscheinlichkeit': regen_prozent,
        'mittlere_wahrscheinlichkeit': mittlere_wahrscheinlichkeit,
        'std_wahrscheinlichkeit': std_wahrscheinlichkeit,
        'simulationen': regen_wahrscheinlichkeiten,
        'luftdruck_simuliert': simulierte_luftdruck,
        'luftfeuchtigkeit_simuliert': simulierte_luftfeuchtigkeit
    }

def berechne_regen_score(luftdruck, luftfeuchtigkeit):
    # Normalisierung der Werte (0-1 Skala)
    # Luftdruck: niedrigere Werte = höhere Regenwahrscheinlichkeit
    if luftdruck < 980:
        druck_score = 0.9
    elif luftdruck < 1000:
        druck_score = 0.7
    elif luftdruck < 1013:
        druck_score = 0.5
    elif luftdruck < 1025:
        druck_score = 0.3
    else:
        druck_score = 0.1
    
    # Luftfeuchtigkeit: höhere Werte = höhere Regenwahrscheinlichkeit
    if luftfeuchtigkeit > 85:
        feuchte_score = 0.9
    elif luftfeuchtigkeit > 75:
        feuchte_score = 0.7
    elif luftfeuchtigkeit > 65:
        feuchte_score = 0.5
    elif luftfeuchtigkeit > 50:
        feuchte_score = 0.3
    else:
        feuchte_score = 0.1
    
    # Gewichtete Kombination (Luftfeuchtigkeit wichtiger)
    regen_score = 0.4 * druck_score + 0.6 * feuchte_score
    
    # Zufälligkeit hinzufügen für realistische Variation
    regen_score += np.random.normal(0, 0.1)
    
    # Auf [0,1] begrenzen
    return np.clip(regen_score, 0, 1)

def erstelle_visualisierung(regen_wahrscheinlichkeiten, luftdruck, luftfeuchtigkeit, 
                          regen_prozent, vorhersage_datum, zeitraum_text):
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    datum_str = vorhersage_datum.strftime('%d.%m.%Y')
    wochentag = ['Montag', 'Dienstag', 'Mittwoch', 'Donnerstag', 'Freitag', 'Samstag', 'Sonntag'][vorhersage_datum.weekday()]
    
    fig.suptitle(f'Monte-Carlo-Simulation: Regenvorhersage Basel\n{wochentag}, {datum_str} ({zeitraum_text})', 
                 fontsize=16, fontweight='bold')
    
    # 1. Histogramm der Regenwahrscheinlichkeiten
    axes[0,0].hist(regen_wahrscheinlichkeiten * 100, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
    axes[0,0].axvline(x=50, color='red', linestyle='--', label='Regen-Schwelle (50%)')
    axes[0,0].axvline(x=regen_prozent, color='orange', linestyle='-', linewidth=2, 
                     label=f'Prognose: {regen_prozent:.1f}%')
    axes[0,0].set_xlabel('Regenwahrscheinlichkeit (%)')
    axes[0,0].set_ylabel('Häufigkeit')
    axes[0,0].set_title(f'Verteilung der Regenwahrscheinlichkeiten\nfür {datum_str}')
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)
    
    # 2. Scatter Plot: Luftdruck vs Luftfeuchtigkeit
    scatter = axes[0,1].scatter(luftdruck, luftfeuchtigkeit, 
                              c=regen_wahrscheinlichkeiten, cmap='RdYlBu_r', 
                              alpha=0.6, s=1)
    axes[0,1].set_xlabel('Luftdruck (hPa)')
    axes[0,1].set_ylabel('Relative Luftfeuchtigkeit (%)')
    axes[0,1].set_title(f'Wetterbedingungen und Regenwahrscheinlichkeit\n{zeitraum_text}')
    plt.colorbar(scatter, ax=axes[0,1], label='Regenwahrscheinlichkeit')
    
    # 3. Kumulativ Verteilung
    sorted_probs = np.sort(regen_wahrscheinlichkeiten * 100)
    y_vals = np.arange(1, len(sorted_probs) + 1) / len(sorted_probs) * 100
    axes[1,0].plot(sorted_probs, y_vals, linewidth=2, color='green')
    axes[1,0].axvline(x=regen_prozent, color='red', linestyle='--', 
                     label=f'Regen-Wahrscheinlichkeit: {regen_prozent:.1f}%')
    axes[1,0].set_xlabel('Regenwahrscheinlichkeit (%)')
    axes[1,0].set_ylabel('Kumulative Häufigkeit (%)')
    axes[1,0].set_title('Kumulative Verteilung')
    axes[1,0].legend()
    axes[1,0].grid(True, alpha=0.3)
    
    # 4. Box Plot der Ergebnisse
    box_data = [regen_wahrscheinlichkeiten[regen_wahrscheinlichkeiten <= 0.5] * 100,
                regen_wahrscheinlichkeiten[regen_wahrscheinlichkeiten > 0.5] * 100]
    axes[1,1].boxplot(box_data, labels=['Kein Regen\n(≤50%)', 'Regen\n(>50%)'])
    axes[1,1].set_ylabel('Regenwahrscheinlichkeit (%)')
    axes[1,1].set_title(f'Vergleich: Regen vs. Kein Regen\nfür {wochentag}, {datum_str}')
    axes[1,1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Beispiel-Verwendung (ersetzen Sie die Parameter mit Ihren tatsächlichen Werten)
if __name__ == "__main__":
    # WICHTIG: Ersetzen Sie diese Werte mit den Parametern aus Ihrer verteilung_bestimmen() Funktion
    
    # Beispiel-Parameter (müssen durch Ihre echten Werte ersetzt werden)
    beispiel_params = {
        'a_luftdruck': a_luftdruck,
        'b_luftdruck': b_luftdruck,
        'loc_luftdruck': loc_luftdruck,
        'scale_luftdruck': scale_luftdruck,
        'a_luftfeuchtigkeit': a_luftfeuchtigkeit,
        'b_luftfeuchtigkeit': b_luftfeuchtigkeit,
        'loc_luftfeuchtigkeit': loc_luftfeuchtigkeit,
        'scale_luftfeuchtigkeit': scale_luftfeuchtigkeit
    }
    
    print("Die Parameter stammen aus Ihrer verteilung_bestimmen() Funktion.\n")
    
    # Monte-Carlo-Simulation ausführen (Standard: morgen)
    ergebnisse = monte_carlo_regenvorhersage(**beispiel_params)

# Abhängigkeiten und Unsicherheiten

# Erweiterung durch neue Erkenntnisse